# einops-rearrange — ex8: patchify ↔ unpatchify round-trip

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-rearrange`. Running the final beacon cell reports progress against the `Einops: Rearrange` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Rearrange` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-rearrange`** (exercise 8). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-rearrange"
DD_SUBTOPIC = "Einops: Rearrange"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einops.rearrange — quick refresher

`rearrange(tensor, pattern, **axes_lengths)` is one operator with three jobs: **reorder** axes (`'h w -> w h'`), **compose** them (`'h w c -> (h w) c'`), and **decompose** them (`'(b1 b2) c -> b1 b2 c'`, with `b1=` or `b2=`). Every identifier on the right must appear on the left and vice versa.

The exercises below build on that: each one runs `rearrange` inside a small pipeline where you have to *see* what the layout did — by plotting it, by printing the shape at each step, or by combining 2–3 patterns into a single ML-adjacent transformation.

### Exercise 8 — patchify ↔ unpatchify round-trip

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Compose a ViT-style patchify with its inverse (unpatchify) and verify the round-trip is bit-identical, printing the shape at each of the 4 conceptual stages.
> Keywords: patching, round-trip, shape-debug, ViT
> ```

**KCs targeted:** `rearrange-axis-decomposition`, `rearrange-axis-composition`

ViT-style patch embedding turns an `(N, C, H, W)` image into a sequence of flattened patches `(N, num_patches, patch_dim)`. The *inverse* — unpatchify — is what you need for things like Masked Autoencoders, segmentation decoders, or visualizing what a patch embedding actually looks like.

Implement two functions, both using only `rearrange`:

**`ex8_patchify(img, p)`** — input `(N, C, H, W)`, patch size `p`. Decompose `H -> (h p)` and `W -> (w p)`, then compose `h w` into a single `num_patches` axis and `c p p` into a single `patch_dim` axis. Print the shape at each of the 4 conceptual stages (input → after decompose → after num_patches compose → after patch_dim compose). Return the final `(N, h*w, c*p*p)` tensor.

**`ex8_unpatchify(patches, p, h, w)`** — take `(N, h*w, c*p*p)` and reverse the whole pipeline back to `(N, C, H, W)`. One `rearrange` call is enough.

The test cell composes them and asserts the round-trip is exactly equal to the original image. If you got the patch order wrong on either side, the round-trip will scramble the image.

In [ ]:
def ex8_patchify(img: Tensor, p: int) -> Tensor:
    """(N, C, H, W) -> (N, num_patches, patch_dim) with shape-stage prints."""
    raise NotImplementedError()


def ex8_unpatchify(patches: Tensor, p: int, h: int, w: int) -> Tensor:
    """(N, h*w, c*p*p) -> (N, C, H, W). Inverse of ex8_patchify."""
    raise NotImplementedError()


def _test_ex8():
    N, C, H, W, p = 2, 3, 8, 8, 4
    img = t.arange(N * C * H * W).reshape(N, C, H, W).float()
    patches = ex8_patchify(img, p)
    h, w = H // p, W // p
    assert patches.shape == (N, h * w, C * p * p), (
        f'patchify shape: {patches.shape}, expected {(N, h * w, C * p * p)}'
    )
    recon = ex8_unpatchify(patches, p, h, w)
    assert recon.shape == img.shape, f'unpatchify shape: {recon.shape}'
    assert t.equal(recon, img), 'round-trip should be bit-identical'
    # Sanity: a non-rectangular grid should also work.
    img2 = t.randn(1, 3, 8, 16)
    patches2 = ex8_patchify(img2, p=4)
    recon2 = ex8_unpatchify(patches2, p=4, h=2, w=4)
    assert t.allclose(recon2, img2), 'non-square round-trip failed'
    _dd_passed.add('ex8')
    print("ex8 ✓")

_test_ex8()

<details><summary>Solution</summary>

```python
def ex8_patchify(img: Tensor, p: int) -> Tensor:
    N, C, H, W = img.shape
    h, w = H // p, W // p
    print(f'stage 0 input              : {tuple(img.shape)}  (N, C, H, W)')
    print(f'stage 1 after H,W decompose: ({N}, {C}, {h}, {p}, {w}, {p})  (N, C, h, p, w, p)')
    print(f'stage 2 num_patches compose: ({N}, {h * w}, {C}, {p}, {p})  (N, h*w, C, p, p)')
    print(f'stage 3 patch_dim compose  : ({N}, {h * w}, {C * p * p})  (N, h*w, C*p*p)')
    return rearrange(img, 'n c (h p1) (w p2) -> n (h w) (c p1 p2)', p1=p, p2=p)


def ex8_unpatchify(patches: Tensor, p: int, h: int, w: int) -> Tensor:
    return rearrange(
        patches, 'n (h w) (c p1 p2) -> n c (h p1) (w p2)',
        h=h, w=w, p1=p, p2=p,
    )
```

**Why this is the canonical round-trip test.** In ViT/MAE code, patchify is everywhere but unpatchify hides bugs that don't surface in shape — they surface in *content*. The `t.equal(recon, img)` assertion is the only way to catch e.g. a swapped `p1`/`p2` binding, or `(h w)` written as `(w h)` on the inverse.

**Why the stage prints?** When you debug a real patchify bug, you'll mentally walk these four stages anyway. Doing it on screen makes the failure mode obvious: if stage-3 patch_dim isn't `C*p*p`, your last compose is wrong.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex8'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex8',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()